In [7]:
print("hello world")

hello world


In [8]:
df_lib = df[df["code_location"] != "developer_written"].copy()

small_libs = ["SocialMedia", "Analytics", "Cloud"]
df_lib["lib_grouped"] = df_lib["code_location"].replace(small_libs, "Other")

print(df_lib["lib_grouped"].value_counts())


NameError: name 'df' is not defined

In [ ]:
import numpy as np
from sklearn.preprocessing import OneHotEncoder

Y = df_lib["verdict"].values
T = df_lib["is_third_party"].values             # still binary
X = OneHotEncoder(drop="first").fit_transform(df_lib[["lib_grouped"]]).toarray()

cate_lib = CausalForestDML(
    model_y=RandomForestRegressor(),
    model_t=RandomForestClassifier(),
    n_estimators=500, random_state=42
)
cate_lib.fit(Y, T, X=X)

df_lib["cate_library"] = cate_lib.effect(X)

lib_effects = df_lib.groupby("lib_grouped")["cate_library"].mean().sort_values()
print(lib_effects)


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
sns.barplot(x=lib_effects.index, y=lib_effects.values)
plt.xticks(rotation=45)
plt.ylabel("Mean Conditional Average Treatment Effect (CATE)")
plt.title("Heterogeneous Impact of Library Type on Alert Precision")
plt.show()
